# (기초) LLM + 위젯으로 “전화 상담 챗봇” 최소 구현

이 노트북은 다음을 최소 코드로 실습합니다.

- Hugging Face `Qwen2.5-1.5B-Instruct` 모델 로드
- `SYSTEM_PROMPT`로 상담자 규칙 부여(존댓말/질문 1개/조언 금지)
- ipywidgets(Text + Button + Output)로 간단 UI 구성
- 대화 로그를 CSV로 저장

> 수업에서는 "기초 동작"만 구현하고, 이후 개선 아이디어는 팀별로 확장합니다.

## 1) 라이브러리 불러오기

- 모델 실행: `torch`, `transformers`
- UI: `ipywidgets`
- 로그 저장: `csv`, `datetime`, `uuid`
- 파일 존재 여부 확인: `os`


In [8]:
import os
import torch
import csv
from datetime import datetime
from uuid import uuid4

import ipywidgets as widgets
from IPython.display import display

from transformers import AutoTokenizer, AutoModelForCausalLM

## 2) 모델 로드 (Qwen2.5-1.5B-Instruct)

- GPU가 있으면 FP16 + device_map="auto"로 로드
- 없으면 CPU(FP32)로 로드
- `model.eval()`로 추론 모드로 전환

In [9]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,  # ✅ torch_dtype
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model.eval()

print("Model loaded:", MODEL_ID)
print("Device:", next(model.parameters()).device)

Model loaded: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0


## 3) 상담자 규칙(System Prompt)

이 프롬프트가 "상담자의 말투/규칙"을 결정합니다.

핵심 규칙:
- 상담자가 먼저 시작
- 한국어 존댓말
- 질문은 1개만
- 공감 1문장 + 질문 1개
- 조언/해결책/의학 판단 금지
- 특정 표현이면 종료


In [10]:
SYSTEM_PROMPT = """
당신은 AI 안부 확인 전화 상담자입니다.

규칙:
- 반드시 상담자가 먼저 말을 시작합니다.
- 한국어 존댓말만 사용합니다.
- 한 번에 질문은 반드시 1개만 합니다.
- 사용자가 답하면:
  (1) 짧은 공감 1문장
  (2) 다음 질문 1개
- 조언, 해결책, 의학적 판단은 하지 않습니다.
- 통화는 2~3분 분량입니다.

종료 규칙:
- 사용자가 "지금 바빠요", "나중에요", "그만" 등의 표현을 하면
  → 정중히 종료 인사 후 질문 없이 종료합니다.
"""

## 4) LLM 호출 함수(chat_llm)

- history(messages)를 받아서 chat template로 프롬프트 생성
- `model.generate()`로 답변 생성
- 생성된 답변을 history에 누적

In [11]:
def chat_llm(history, user_text=None, max_new_tokens=120):
    if user_text:
        history.append({"role": "user", "content": user_text})

    prompt = tokenizer.apply_chat_template(
        history,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # pad_token 처리(모델/토크나이저에 따라 필요)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.4,
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    reply = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    history.append({"role": "assistant", "content": reply})
    return reply

## 5) CSV 로그 저장

- session_id 단위로 대화를 기록합니다.
- 수업에서는 "저장되는 구조를 이해"하는 것이 목표입니다.

In [12]:
CSV_PATH = "chat_log.csv"

def save_log(session_id, user, assistant):
    is_new = not os.path.exists(CSV_PATH)

    with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if is_new:
            writer.writerow(["session_id", "timestamp", "user", "assistant"])

        writer.writerow([
            session_id,
            datetime.now().isoformat(timespec="seconds"),
            user,
            assistant
        ])

## 6) 위젯 UI(SimpleChatUI)

구성:
- Output: 대화 출력 영역
- Text: 사용자 입력
- Button: 보내기

Enter 전송:
- `Text.on_submit`는 deprecated
- `continuous_update=False + observe(value)`로 Enter/확정 이벤트를 처리합니다.

중복 전송 방지:
- observe 이벤트가 여러 번 들어오는 경우를 막기 위해 `_last_sent_value`를 사용합니다.

In [13]:
class SimpleChatUI:
    def __init__(self):
        self.output = widgets.Output(
            layout={"border": "1px solid #ccc", "height": "300px", "overflow_y": "auto"}
        )

        self.input = widgets.Text(
            placeholder="답변을 입력하세요",
            layout=widgets.Layout(width="80%")
        )
        self.send_btn = widgets.Button(description="보내기")

        self.box = widgets.VBox([
            self.output,
            widgets.HBox([self.input, self.send_btn])
        ])

        # 버튼 클릭 전송
        self.send_btn.on_click(self.on_send)

        # ✅ on_submit 대체: observe 방식
        self.input.continuous_update = False
        self.input.observe(self._on_text_commit, names="value")
        self._last_sent_value = ""

        # state
        self.history = [{"role": "system", "content": SYSTEM_PROMPT}]
        self.session_id = str(uuid4())[:8]

        # 상담 시작 멘트 (AI가 먼저 시작)
        opening = "안녕하세요. 오늘 안부 확인 전화드리는 AI 상담 도우미입니다. 지금 2~3분 통화 괜찮으실까요?"
        self.history.append({"role": "assistant", "content": opening})
        self._print("챗봇", opening)

    def _print(self, who, text):
        with self.output:
            print(f"{who}: {text}")

    def _on_text_commit(self, change):
        new_val = (change["new"] or "").strip()

        # 비어있거나, 이미 보낸 값이면 무시(중복 이벤트 방지)
        if not new_val or new_val == self._last_sent_value:
            return

        self._last_sent_value = new_val
        self.on_send()

    def on_send(self, _=None):
        user_text = (self.input.value or "").strip()
        if not user_text:
            return

        # 중복 방지(버튼/Enter 경로 통합)
        if user_text != self._last_sent_value:
            self._last_sent_value = user_text

        # UI 즉시 반영
        self.input.value = ""

        self._print("사용자", user_text)

        # 종료 키워드
        if user_text in ("그만", "나중에요", "지금 바빠요"):
            end_msg = "알겠습니다. 오늘 말씀 감사합니다. 다음에 다시 연락드릴게요."
            self._print("챗봇", end_msg)
            return

        reply = chat_llm(self.history, user_text)
        self._print("챗봇", reply)
        save_log(self.session_id, user_text, reply)

    def render(self):
        display(self.box)

## 7) 실행

아래 셀을 실행하면 위젯 UI가 나타납니다.

In [14]:
ui = SimpleChatUI()
ui.render()